# Credit Risk Portfolio Analysis

A reproducible review of the synthetic portfolio behind the monitoring dashboard. The notebook validates its input, summarizes the latest reporting month, and examines portfolio trends, delinquency, defaults, and customer concentration.

**Dataset:** `data/processed/fact_exposure_snapshot.csv`  
**Grain:** one row per customer and reporting month.

## 1. Setup and portable data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent containing the processed data directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/processed. Run this notebook inside the repository."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "fact_exposure_snapshot.csv"
print(f"Dataset: {DATA_PATH.relative_to(PROJECT_ROOT)}")

## 2. Validate the dataset

These fail-fast checks prevent incomplete or malformed data from producing misleading metrics.

In [ ]:
REQUIRED_COLUMNS = {
    "customer_id", "month_key", "exposure_amount",
    "overdue_amount", "default_flag", "open_invoices",
}
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Processed exposure file not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
missing_columns = REQUIRED_COLUMNS.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df["month_key"] = pd.to_datetime(df["month_key"], format="%Y-%m", errors="raise")
numeric_columns = ["exposure_amount", "overdue_amount", "open_invoices"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="raise")

if df["default_flag"].dtype != bool:
    default_text = df["default_flag"].astype(str).str.strip().str.lower()
    if not default_text.isin({"true", "false"}).all():
        raise ValueError("default_flag must contain only True or False.")
    df["default_flag"] = default_text.eq("true")

duplicate_count = int(df.duplicated(["customer_id", "month_key"]).sum())
if duplicate_count:
    raise ValueError(f"Found {duplicate_count:,} duplicate customer-month records.")
if (df[numeric_columns] < 0).any().any():
    raise ValueError("Negative values found in non-negative financial fields.")

df = df.sort_values(["month_key", "customer_id"]).reset_index(drop=True)
df.head()

## 3. Coverage and data quality

In [ ]:
coverage = pd.Series({
    "Rows": len(df),
    "Customers": df["customer_id"].nunique(),
    "Reporting months": df["month_key"].nunique(),
    "First month": df["month_key"].min().strftime("%Y-%m"),
    "Latest month": df["month_key"].max().strftime("%Y-%m"),
    "Duplicate customer-month rows": duplicate_count,
    "Missing values": int(df.isna().sum().sum()),
}, name="Value")
coverage.to_frame()

## 4. Latest-month portfolio snapshot

The reporting month is selected from the data rather than hard-coded.

In [ ]:
latest_month = df["month_key"].max()
latest = df.loc[df["month_key"].eq(latest_month)].copy()
total_exposure = latest["exposure_amount"].sum()
total_overdue = latest["overdue_amount"].sum()
defaulted_exposure = latest.loc[latest["default_flag"], "exposure_amount"].sum()

latest_kpis = pd.Series({
    "Reporting month": latest_month.strftime("%Y-%m"),
    "Active customers": latest["customer_id"].nunique(),
    "Total exposure": total_exposure,
    "Overdue amount": total_overdue,
    "Overdue rate (%)": 100 * total_overdue / total_exposure if total_exposure else np.nan,
    "Defaulted customers": int(latest["default_flag"].sum()),
    "Defaulted exposure": defaulted_exposure,
    "Defaulted exposure rate (%)": 100 * defaulted_exposure / total_exposure if total_exposure else np.nan,
    "Open invoices": latest["open_invoices"].sum(),
}, name="Value")
latest_kpis.to_frame()

## 5. Portfolio trend

In [ ]:
monthly = (
    df.groupby("month_key", as_index=False)
    .agg(
        customers=("customer_id", "nunique"),
        total_exposure=("exposure_amount", "sum"),
        overdue_amount=("overdue_amount", "sum"),
        defaulted_customers=("default_flag", "sum"),
        open_invoices=("open_invoices", "sum"),
    )
)
monthly["overdue_rate_pct"] = 100 * monthly["overdue_amount"] / monthly["total_exposure"]
monthly["exposure_change_pct"] = monthly["total_exposure"].pct_change() * 100
monthly.round(2)

## 6. Delinquency and concentration

The delinquency bands are analytical monitoring categories, not regulatory classifications.

In [ ]:
latest["overdue_rate"] = np.where(
    latest["exposure_amount"].gt(0),
    latest["overdue_amount"] / latest["exposure_amount"],
    np.nan,
)
latest["delinquency_band"] = pd.cut(
    latest["overdue_rate"],
    bins=[-np.inf, 0, 0.05, 0.15, 0.30, np.inf],
    labels=["Current", "Low (0-5%)", "Moderate (5-15%)", "High (15-30%)", "Severe (>30%)"],
)
delinquency_summary = (
    latest.groupby("delinquency_band", observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        exposure=("exposure_amount", "sum"),
        overdue_amount=("overdue_amount", "sum"),
        defaulted_customers=("default_flag", "sum"),
    )
    .reset_index()
)
delinquency_summary["exposure_share_pct"] = 100 * delinquency_summary["exposure"] / total_exposure
delinquency_summary.round(2)

In [ ]:
customer_concentration = (
    latest[["customer_id", "exposure_amount", "overdue_amount", "default_flag"]]
    .sort_values("exposure_amount", ascending=False)
    .rename(columns={"exposure_amount": "exposure"})
)
customer_concentration["portfolio_share_pct"] = 100 * customer_concentration["exposure"] / total_exposure
top_10_share = customer_concentration.head(10)["portfolio_share_pct"].sum()
print(f"Top-10 customer exposure share: {top_10_share:.2f}%")
customer_concentration.head(10).round(2)

## 7. Interpretation checklist

- Is exposure growth accompanied by a rising overdue rate?
- Are defaults isolated or persistent across reporting months?
- How much exposure is in the high and severe delinquency bands?
- Does the top-10 customer share indicate material concentration?
- Could missing data or coverage changes distort the dashboard?

The dataset is synthetic. Results are for demonstration, not lending or regulatory decisions.